# Haplotype clustering and network analysis (Ag3)

This notebook demonstrates three `Ag3` methods for exploring haplotype-level
(phased) genetic variation: `plot_haplotype_clustering`,
`plot_haplotype_network`, and `haplotype_pairwise_distances`.

All three work on *phased haplotypes* — reconstructed from diploid genotype
calls via one of the resource's phasing analyses (e.g. `"gamb_colu"`) — so
each sample contributes two haplotypes.

In [1]:
import malariagen_data
ag3 = malariagen_data.Ag3(
    "simplecache::gs://vo_agam_release_master_us_central1",
    simplecache=dict(cache_storage="../../gcs_cache"),
    results_cache="../../results_cache",
)
ag3

/opt/homebrew/Caskroom/miniconda/base/envs/malariagen2/lib/python3.11/site-packages/requests/__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


<MalariaGEN Ag3 API client>
Storage URL                           : simplecache::gs://vo_agam_release_master_us_central1
Data releases available               : 3.0, 3.1, 3.2, 3.3, 3.4, 3.5, 3.6, 3.7, 3.8, 3.9, 3.10, 3.11, 3.12, 3.13, 3.14, 3.15, 3.16
Results cache                         : /Users/katie.barr/malariagen-data-python/results_cache
Cohorts analysis                      : 20260120
AIM analysis                          : 20220528
Site filters analysis                 : dt_20200416
Software version                      : malariagen_data 15.8.0.post13+b769b728
Client location                       : England, United Kingdom
Data filtered to unrestricted use only: False
Data filtered to surveillance use only: False
Relevant data releases                : 3.0, 3.1, 3.2, 3.3, 3.4, 3.5, 3.6, 3.7, 3.8, 3.9, 3.10, 3.11, 3.12, 3.13, 3.14, 3.15, 3.16
---
Please note that data are subject to terms of use,
for more information see the Vector Observatory website https://www.malariagen.net/vobs/
or contact support@malariagen.net. For API documentation see 
https://malariagen.github.io/malariagen-data-python/v15.8.0.post13+b769b728/Ag3.html

## `haplotype_pairwise_distances`

Computes the pairwise distance between every pair of haplotypes in a region,
underpinning both `plot_haplotype_clustering` and (indirectly) haplotype
network construction.

Parameters:
- **region**: genome region(s) to compute haplotypes over (can be a single
  region or a sequence of regions).
- **distance_metric**: `"hamming"` (raw count of differing SNPs between two
  haplotypes) or `"dxy"` (Hamming distance normalised by the total number
  of sites in the region, giving nucleotide diversity per site).
- **analysis**: which haplotype phasing analysis to use (e.g.
  `"gamb_colu"`, `"gamb_colu_arab"`, `"arab"` — see `phasing_analysis_ids`).
  This determines which reference panel/sample subset is eligible for
  phasing.
- **sample_sets**, **sample_query**, **sample_query_options**: standard
  sample-selection parameters.
- **cohort_size**: randomly down-sample to this many samples if more are
  available (raises an error if fewer).
- **random_seed**: seed for reproducible down-sampling.
- **chunks**, **inline_array**: Dask/Zarr loading controls.
- **return_dataset**: if `False` (default), returns a
  `(dist, phased_samples, n_snps)` tuple, where `dist` is a condensed
  (1-D, `scipy.spatial.distance.squareform`-style) pairwise distance array.
  If `True`, instead returns an `xarray.Dataset` bundling the square-form
  distance matrix with haplotype sample labels and the SNP count as
  metadata — more convenient for downstream xarray-based analysis. We use
  `return_dataset=True` here to show that form.

In [2]:
ds_dist = ag3.haplotype_pairwise_distances(
    region="2L:2,410,000-2,430,000",
    sample_sets=["AG1000G-GH", "AG1000G-BF-B"],
    analysis="gamb_colu",
    distance_metric="hamming",
    return_dataset=True,
)
ds_dist

<xarray.Dataset> Size: 646kB
Dimensions:    (sample_x: 398, sample_y: 398)
Coordinates:
    sample_id  (sample_x) <U8 13kB 'AB0326-C' 'AB0326-C' ... 'AA0141-C'
Dimensions without coordinates: sample_x, sample_y
Data variables:
    dist       (sample_x, sample_y) int32 634kB 0 32 31 1 31 31 ... 0 0 0 0 1 0
Attributes:
    n_snps:   20001

## `plot_haplotype_clustering`

Hierarchically clusters haplotypes in a region (using
`haplotype_pairwise_distances` internally) and renders the result as an
interactive Plotly dendrogram, with each leaf representing one haplotype
(so two leaves per diploid sample) coloured/symbolised by sample metadata.

Parameters:
- **region**: genome region(s) to cluster haplotypes over.
- **analysis**: haplotype phasing analysis to use, as above.
- **sample_sets**, **sample_query**, **sample_query_options**: sample
  selection.
- **cohort_size**, **random_seed**: optional down-sampling, as above.
- **color**, **symbol**: sample metadata column name (or dict of custom
  label -> query) used to colour/symbolise dendrogram leaves.
- **linkage_method**: the hierarchical clustering linkage algorithm (one of
  `"single"`, `"complete"`, `"average"`, `"weighted"`, `"centroid"`,
  `"median"`, `"ward"` — see `scipy.cluster.hierarchy.linkage`). `"single"`
  (nearest-neighbour) is the default for haplotype clustering; `"complete"`
  (farthest-neighbour) tends to produce more compact, evenly-sized clusters.
- **distance_metric**: `"hamming"` or `"dxy"`, as above.
- **count_sort** / **distance_sort**: control dendrogram leaf ordering (for
  each internal node, sort children by descendant count or by distance).
  Mutually exclusive; if neither is given, `count_sort=True` is used.
- **title**, **title_font_size**: plot title text/size. If `title=True`
  (default), a title is auto-built from the sample sets/query/region.
- **width**, **height**: figure size in pixels.
- **show**: if `True` (default), displays the figure; if `False`, returns a
  dict with the figure and underlying clustering data (`n_snps`, `dist`,
  `dist_samples`, `leaf_data`).
- **renderer**: Plotly renderer name to use for display.
- **render_mode**: `"auto"`, `"svg"`, or `"webgl"` — rendering backend.
- **leaf_y**: y-coordinate at which leaf markers are drawn.
- **marker_size**, **line_width**, **line_color**: leaf marker and
  dendrogram line styling.
- **color_discrete_sequence**, **color_discrete_map**, **category_orders**:
  fine control over the colour palette and legend ordering.
- **legend_sizing**: `"constant"` or `"trace"` — whether legend marker size
  scales with the plotted marker size.
- **chunks**, **inline_array**: Dask/Zarr loading controls.

We use a small ~20 kbp region and two sample sets, with `linkage_method`
and `distance_metric` set to non-default values to illustrate their
effect.

In [3]:
ag3.plot_haplotype_clustering(
    region="2L:2,410,000-2,430,000",
    sample_sets=["AG1000G-GH", "AG1000G-BF-B"],
    analysis="gamb_colu",
    color="taxon",
    symbol="country",
    linkage_method="complete",
    distance_metric="dxy",
    count_sort=True,
    distance_sort=False,
    width=900,
    height=500,
)

Load sample metadata: ⠋ (0:00:00.00)

Access haplotypes: ⠋ (0:00:00.00)

Access haplotypes: ⠙ (0:00:00.09)

Access haplotypes: ⠹ (0:00:00.17)

Load haplotypes:   0%|          | 0/9 [00:00<?, ?it/s]

...

Save results to cache: ⠋ (0:00:00.00)

Plot dendrogram: ⠋ (0:00:00.00)

Plot dendrogram: ⠙ (0:00:00.09)

Plot dendrogram: ⠹ (0:00:00.19)

{'figure': Figure({
     'data': [{'hovertemplate': 'x=%{x}<br>y=%{y}<extra></extra>',
               'legendgroup': '',
               'line': {'color': 'black', 'dash': 'solid', 'width': 0.5},
               'marker': {'size': 5, 'symbol': 'circle'},
               'mode': 'lines',
               'name': '',
               'orientation': 'v',
               'showlegend': False,
               'type': 'scatter',
               'x': {'bdata': ('AAAAAAAAHEAAAAAAAAAcQAAAAAAAAC' ... 'Br0URAAAAAQGvRREAAAAAAAAD4fw=='),
                     'dtype': 'f8'},
               'xaxis': 'x',
               'y': {'bdata': ('AAAAAAAAAAAAAAAAAAAAAAAAAAAAAA' ... 'MYYmA/TqR7xOmoQz8AAAAAAAD4fw=='),
                     'dtype': 'f8'},
               'yaxis': 'y'},
              {'customdata': array([['AA0107-C_0', 'Koforidua_D6', 'AG1000G-GH', ..., 'Koforidua', 2012,
                                     11],
                                    ['AA0123-C_0', 'Twifo_Praso__D2', 'AG1000G-GH', ..., 'Twifo P

## `plot_haplotype_network`

Constructs a median-joining haplotype network and displays it as an
interactive Cytoscape graph (via Plotly Dash). Each node is a *distinct*
observed haplotype; node size (area) scales with how many times that exact
haplotype was observed; an edge between two nodes means those haplotypes
differ by (up to) one SNP (or more if a component would otherwise be
disconnected — see `max_dist`).

Parameters:
- **region**: genome region(s) to build haplotypes from.
- **analysis**: haplotype phasing analysis to use.
- **sample_sets**, **sample_query**, **sample_query_options**: sample
  selection.
- **max_dist**: maximum number of SNP differences allowed when joining
  otherwise-disconnected network components (default 2). Larger values
  merge more distantly-related haplotypes into the same network at the
  cost of a busier graph.
- **color**: either a sample metadata column name (optionally with a
  `"cohorts_"` prefix implied, e.g. `"admin1_iso"`), or a dict mapping
  custom group labels to sample-metadata queries. Each node is rendered as
  a pie chart showing the mix of haplotypes' group membership. `None`
  disables colouring.
- **color_discrete_sequence**, **color_discrete_map**, **category_orders**:
  colour palette controls, as above.
- **node_size_factor**: scales node size relative to haplotype observation
  count (default 50); increase to make count differences more visually
  prominent.
- **layout**: name of the Cytoscape graph layout algorithm used to
  position nodes (default `"cose"`, a force-directed layout).
- **layout_params**: extra keyword parameters passed to the chosen layout
  algorithm.
- **server_port**: manually pin the local port the Dash app runs on
  (otherwise chosen automatically); useful to avoid clashes when running
  multiple network plots in one session.
- **server_mode**: how the Dash app is launched in Jupyter — `"inline"`
  (default, embeds an iframe in the notebook output), `"external"`,
  `"jupyterlab"`, or `"tab"`.
- **height**, **width**: size of the Dash app/plot area.
- **serve_scripts_locally**: if `True` (default), serve JS/CSS assets
  locally rather than from a CDN.
- **chunks**, **inline_array**: Dask/Zarr loading controls.

We restrict to a single sample set and taxon (`coluzzii` samples from
`"3.0"`) to keep the network small and readable, and colour nodes by
`"country"`.

In [4]:
ag3.plot_haplotype_network(
    region="2L:2,358,158-2,431,617",
    analysis="gamb_colu",
    sample_sets="3.0",
    sample_query="taxon == 'coluzzii'",
    color="country",
    max_dist=2,
)

Access haplotypes: ⠋ (0:00:00.00)

Access haplotypes: ⠙ (0:00:00.09)

Access haplotypes: ⠹ (0:00:00.18)

Access haplotypes: ⠸ (0:00:00.26)

Access haplotypes: ⠼ (0:00:00.35)

...

Compute haplotype network: ⠼ (0:00:02.35)

Compute haplotype network: ⠴ (0:00:02.44)

Compute haplotype network: ⠦ (0:00:02.53)

Compute haplotype network: ⠧ (0:00:02.62)

Compute haplotype network: ⠇ (0:00:02.73)